# News Article Sentiment Analysis Pipeline

This notebook processes news article titles and uses the FIN-RoBERTa model to calculate sentiment scores. 

**Input**: CSV files with columns (headline, date) - format: `TICKER_news.csv`

**Output**: Daily sentiment + price data per ticker - format: `TICKER_news_sentiment_daily.csv`

**Model**: FIN-RoBERTa (alasteirho/FIN-RoBERTa-Custom) - Custom financial domain-specific RoBERTa

## 1. Imports and Setup

In [6]:
import os
import re
from pathlib import Path
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from tqdm import tqdm
import yfinance as yf

In [7]:
# Directories definitions
BASE_DIR = Path.cwd().resolve()
if (BASE_DIR / "gdelt_news_data").exists():
    pass
elif (BASE_DIR.parent / "gdelt_news_data").exists():
    BASE_DIR = BASE_DIR.parent

INPUT_DIR = str(BASE_DIR / "gdelt_news_data")
OUTPUT_DIR = BASE_DIR / "processed_data" / "news_sentiment_daily"

print(f"BASE_DIR: {BASE_DIR}")
print(f"INPUT_DIR: {INPUT_DIR}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

BASE_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP
INPUT_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\gdelt_news_data
OUTPUT_DIR: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily


## Exploratory Data Analysis (EDA)

Summary statistics for the news article dataset - total counts and breakdown by ticker.

In [8]:
# EDA: Count articles per ticker and create summary table
import glob

# Get all news CSV files
news_files = glob.glob(os.path.join(INPUT_DIR, "*_news.csv"))

# Build summary data
eda_data = []
total_articles = 0

for file_path in sorted(news_files):
    filename = os.path.basename(file_path)
    ticker = filename.replace("_news.csv", "")
    
    df = pd.read_csv(file_path)
    article_count = len(df)
    total_articles += article_count
    
    # Get date range if available
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'], errors='coerce')
        date_min = df['date'].min().strftime('%Y-%m-%d') if pd.notna(df['date'].min()) else 'N/A'
        date_max = df['date'].max().strftime('%Y-%m-%d') if pd.notna(df['date'].max()) else 'N/A'
    else:
        date_min, date_max = 'N/A', 'N/A'
    
    eda_data.append({
        'Ticker': ticker,
        'Article Count': article_count,
        'Start Date': date_min,
        'End Date': date_max
    })

# Create summary DataFrame
eda_df = pd.DataFrame(eda_data)
eda_df = eda_df.sort_values('Article Count', ascending=False).reset_index(drop=True)

# Display summary table
print("=" * 70)
print("NEWS ARTICLE DATASET SUMMARY")
print("=" * 70)
print(f"\nTotal Number of Tickers: {len(eda_df)}")
print(f"Total Number of Articles: {total_articles:,}")
print(f"Average Articles per Ticker: {total_articles / len(eda_df):,.0f}")
print(f"\nMin Articles: {eda_df['Article Count'].min():,} ({eda_df.loc[eda_df['Article Count'].idxmin(), 'Ticker']})")
print(f"Max Articles: {eda_df['Article Count'].max():,} ({eda_df.loc[eda_df['Article Count'].idxmax(), 'Ticker']})")
print("\n" + "=" * 70)
print("BREAKDOWN BY TICKER")
print("=" * 70)

# Display the table
display(eda_df.style.format({'Article Count': '{:,}'}).set_properties(**{'text-align': 'left'}))

# Also show as percentage of total
eda_df['% of Total'] = (eda_df['Article Count'] / total_articles * 100).round(2)
print("\n" + "=" * 70)
print("ARTICLE DISTRIBUTION (% of Total)")
print("=" * 70)
display(eda_df[['Ticker', 'Article Count', '% of Total']].style.format({
    'Article Count': '{:,}',
    '% of Total': '{:.2f}%'
}).set_properties(**{'text-align': 'left'}))

NEWS ARTICLE DATASET SUMMARY

Total Number of Tickers: 20
Total Number of Articles: 46,017
Average Articles per Ticker: 2,301

Min Articles: 409 (V)
Max Articles: 5,217 (NVDA)

BREAKDOWN BY TICKER


,Ticker,Article Count,Start Date,End Date
0,NVDA,"5,217",2023-10-10,2025-10-10
1,AAPL,"4,367",2023-10-10,2025-10-10
2,GOOGL,"4,097",2023-10-10,2025-10-10
3,TSLA,"3,963",2023-10-10,2025-10-10
4,META,"3,683",2023-10-10,2025-10-10
5,BRK-B,"3,611",2023-10-10,2025-10-10
6,MSFT,"2,767",2023-10-10,2025-10-10
7,AVGO,"2,570",2023-10-10,2025-10-10
8,JPM,"2,494",2023-10-10,2025-10-10
9,AMZN,"2,272",2023-10-10,2025-10-09



ARTICLE DISTRIBUTION (% of Total)


,Ticker,Article Count,% of Total
0,NVDA,"5,217",11.34%
1,AAPL,"4,367",9.49%
2,GOOGL,"4,097",8.90%
3,TSLA,"3,963",8.61%
4,META,"3,683",8.00%
5,BRK-B,"3,611",7.85%
6,MSFT,"2,767",6.01%
7,AVGO,"2,570",5.58%
8,JPM,"2,494",5.42%
9,AMZN,"2,272",4.94%


## 2. Load FIN-RoBERTa Model

In [9]:
# Load FinRoBERTa model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("alasteirho/FIN-RoBERTa-Custom")
model = AutoModelForSequenceClassification.from_pretrained("alasteirho/FIN-RoBERTa-Custom")

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()
print(f"Model loaded onto {device}")

Model loaded onto cuda


## 3. Preprocessing Function

Cleans news article titles using REG-EX by removing:
- URLs
- Ticker symbols (NASDAQ:XXX format)
- Exchange prefixes
- News source metadata
- Special characters

In [10]:
def preprocess_title(title):
    if pd.isna(title) or not isinstance(title, str):
        return None

    title = re.sub(r'\s+', ' ', title).strip()
    title = re.sub(r'http\S+|www\.\S+', '', title)
    title = re.sub(r'\(\s*(NASDAQ|NYSE|AMEX)\s*:\s*\w+\s*\)', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*[-|]\s*[A-Z]{1,5}\s*$', '', title)
    title = re.sub(r'^\s*(NASDAQ|NYSE|AMEX)\s*:\s*', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*\|\s*[A-Z][a-z]+\s*$', '', title)
    title = re.sub(r'\s*-\s*[A-Z][a-z]+\s+[A-Z][a-z]+\s*$', '', title)
    title = re.sub(r'[^\w\s\.,!?\'\"\-]', ' ', title)
    title = re.sub(r'\s+', ' ', title).strip()

    if len(title) < 15:
        return None

    return title

## 4. Sentiment Scoring

Uses FIN-RoBERTa to calculate sentiment scores from -1 (negative) to +1 (positive)

In [11]:
def get_sentiment_score(texts, batch_size=16):
    if not texts:
        return []

    scores = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)
            predictions = torch.softmax(outputs.logits, dim=1)

        # FinRoBERTa outputs: [negative, neutral, positive]
        # Score = P(positive) - P(negative)
        batch_scores = predictions[:, 2] - predictions[:, 0]
        scores.extend(batch_scores.cpu().numpy().tolist())

    return scores

## 5. File Processing Function

In [12]:
def process_file(input_path, ticker):
    print(f"\nProcessing: {os.path.basename(input_path)} (Ticker: {ticker})")

    df = pd.read_csv(input_path)

    if 'headline' not in df.columns or 'date' not in df.columns:
        print(f"Skipping: Missing 'headline' or 'date' column")
        return

    print("Preprocessing news article titles...")
    df['clean_headline'] = df['headline'].apply(preprocess_title)

    valid_df = df.dropna(subset=['clean_headline']).copy()

    if len(valid_df) == 0:
        print("No valid headlines found after preprocessing")
        return

    print(f"Processing {len(valid_df)} valid news headlines")

    headlines = valid_df['clean_headline'].tolist()
    scores = get_sentiment_score(headlines)
    valid_df['sentiment_score'] = scores

    valid_df['datetime'] = pd.to_datetime(valid_df['date'])
    valid_df['sentiment_score'] = valid_df['sentiment_score'].round(4)
    result = valid_df[['datetime', 'sentiment_score']].copy()
    result['ticker'] = ticker
    result = result.sort_values('datetime')
    return result

## 6. Helper Functions

In [13]:
def extract_ticker(filename):
    base = filename.replace('.csv', '')

    if '_news' in base:
        return base.split('_news')[0]
    elif '_' in base:
        parts = base.split('_')
        for part in parts:
            if part.isupper() and 1 <= len(part) <= 5:
                return part

    return None


def fetch_price_data(ticker, start_date, end_date):
    try:
        print(f"  Fetching price data for {ticker}...")
        df = yf.download(
            ticker,
            start=start_date,
            end=end_date,
            auto_adjust=False,
            progress=False
        )

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df.reset_index()

        if df.empty:
            print(f"  Warning: No price data available for {ticker}")
            return None

        if "Date" not in df.columns:
            print(f"  Warning: Date column not found for {ticker}")
            return None

        df["Date"] = pd.to_datetime(df["Date"])
        df["date"] = df["Date"].dt.strftime("%Y-%m-%d")

        price_df = df[["date"]].copy()

        if "Open" in df.columns:
            price_df["open_price"] = pd.to_numeric(df["Open"], errors="coerce")
        else:
            price_df["open_price"] = None

        if "Close" in df.columns:
            price_df["close_price"] = pd.to_numeric(df["Close"], errors="coerce")
        else:
            price_df["close_price"] = None

        return price_df

    except Exception as e:
        print(f"Error fetching price data for {ticker}: {e}")
        return None

## 7. Main Processing Pipeline

Steps:
1. Load news CSV files
2. Extract ticker symbols
3. Preprocess titles and calculate sentiment
4. Aggregate to daily average
5. Fetch price data
6. Merge and save

In [14]:
csv_files = [
    filename 
    for filename in os.listdir(INPUT_DIR) 
    if filename.endswith('.csv')
]

if not csv_files:
    print(f"No CSV files found in {INPUT_DIR}")
else:
    print(f"Found {len(csv_files)} news CSV files to process")

Found 20 news CSV files to process


In [15]:
# Process each news file
results = []
for filename in tqdm(csv_files, desc="Processing news files"):
    input_path = os.path.join(INPUT_DIR, filename)

    ticker = extract_ticker(filename)
    if not ticker:
        print(f"\nSkipping {filename}: Could not extract the ticker symbol")
        continue

    try:
        result = process_file(input_path, ticker)
        if result is not None and not result.empty:
            results.append(result)
    except Exception as e:
        print(f"Error processing {filename}: {e}")

if not results:
    print("\nNo sentiment records generated from news articles.")

Processing news files:   0%|          | 0/20 [00:00<?, ?it/s]


Processing: AAPL_news.csv (Ticker: AAPL)
Preprocessing news article titles...
Processing 4365 valid news headlines


Processing news files:   5%|▌         | 1/20 [00:02<00:53,  2.81s/it]


Processing: AMZN_news.csv (Ticker: AMZN)
Preprocessing news article titles...
Processing 2272 valid news headlines


Processing news files:  10%|█         | 2/20 [00:03<00:33,  1.85s/it]

...

Processing news files:  90%|█████████ | 18/20 [00:24<00:01,  1.05it/s]


Processing: WMT_news.csv (Ticker: WMT)
Preprocessing news article titles...
Processing 1039 valid news headlines


Processing news files:  95%|█████████▌| 19/20 [00:24<00:00,  1.21it/s]


Processing: XOM_news.csv (Ticker: XOM)
Preprocessing news article titles...
Processing 1187 valid news headlines


Processing news files: 100%|██████████| 20/20 [00:25<00:00,  1.28s/it]


In [16]:
# Combine and average daily sentiment
if results:
    combined = pd.concat(results, ignore_index=True)
    combined = combined.dropna(subset=["datetime", "ticker", "sentiment_score"])
    combined = combined.sort_values(["datetime", "ticker"])

    combined["date"] = pd.to_datetime(combined["datetime"]).dt.strftime("%Y-%m-%d")
    total_daily_sentiment = (
        combined.groupby(["date", "ticker"], as_index=False)["sentiment_score"]
        .mean()
        .rename(columns={"sentiment_score": "avg_sentiment"})
        .sort_values(["ticker", "date"])
    )

    print(f"\n Total Daily sentiment records: {len(total_daily_sentiment)}")
    display(total_daily_sentiment.head())


 Total Daily sentiment records: 11388


,date,ticker,avg_sentiment
0,2023-10-10,AAPL,0.295382
17,2023-10-11,AAPL,-0.048833
34,2023-10-12,AAPL,0.203037
54,2023-10-13,AAPL,0.695200
69,2023-10-14,AAPL,-0.073750


## 8. Fetch Price Data and Merge

In [17]:
if results:
    min_date = total_daily_sentiment["date"].min()
    max_date = total_daily_sentiment["date"].max()

    print(f"\nFetching price data for date range: {min_date} to {max_date}")

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    tickers = sorted(total_daily_sentiment["ticker"].unique().tolist())

    for ticker in tickers:
        print(f"\nProcessing ticker: {ticker}")
        
        # 1. Get raw sentiment (only has dates with news)
        ticker_sentiment = total_daily_sentiment[total_daily_sentiment["ticker"] == ticker][["date", "avg_sentiment"]].copy()
        
        # 2. Get full price history (continuous trading days)
        price_data = fetch_price_data(ticker, min_date, max_date)

        if price_data is not None and not price_data.empty:
            # 3. MERGE: Use price_data as the "Left" dataframe (Master List of Dates)
            # This ensures we keep EVERY trading day, even if no news existed
            ticker_data = price_data.merge(ticker_sentiment, on="date", how="left")
            
            # 4. Fill missing sentiment with 0 (Neutral)
            # Assumption: No news = Neutral sentiment
            ticker_data["avg_sentiment"] = ticker_data["avg_sentiment"].fillna(0)
            
            print(f"Merged {len(ticker_data)} records (aligned to Price History)")
        else:
            # Fallback if price download fails
            ticker_data = ticker_sentiment.copy()
            ticker_data["open_price"] = None
            ticker_data["close_price"] = None
            print("No price data available, saving sentiment only (gaps may exist)")

        out_path = OUTPUT_DIR / f"{ticker}_news_sentiment_daily.csv"
        ticker_data.to_csv(out_path, index=False)
        print(f"Saved to: {out_path}")

    print(f"\nCompleted processing for {len(tickers)} tickers")
    print(f"Output directory: {OUTPUT_DIR}")


Fetching price data for date range: 2023-10-10 to 2025-10-10

Processing ticker: AAPL
  Fetching price data for AAPL...
Merged 502 records (aligned to Price History)
Saved to: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily\AAPL_news_sentiment_daily.csv

Processing ticker: AMZN
  Fetching price data for AMZN...
Merged 502 records (aligned to Price History)
Saved to: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily\AMZN_news_sentiment_daily.csv

Processing ticker: AVGO
  Fetching price data for AVGO...
Merged 502 records (aligned to Price History)
Saved to: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily\AVGO_news_sentiment_daily.csv

Processing ticker: BRK-B
  Fetching price data for BRK-B...
Merged 502 records (aligned to Price History)
Saved to: C:\Users\Alasteir\OneDrive - University of Greenwich\FYP\processed_data\news_sentiment_daily\BRK-B_news_sen